# 06 - Team Defensive Profile Clustering

This notebook turns the player-level defensive-action dataset into **team-tournament defensive profiles** and clusters those profiles into tactical families.

## Questions answered
1. Which teams defend with similar phase mixes and spatial footprints?
2. Which clusters are front-foot pressers, deep-block absorbers, balanced teams, or risk-exposed teams?
3. Which team profiles concede the most downstream shot threat (`target_shot_in_10s`) and expected threat (`target_xt_10s`)?
4. Which features most define each cluster?

**Unit of analysis:** one row per `tournament × team`, filtered to teams with enough matches/actions for stable profiles.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (14, 8)
RANDOM_STATE = 42

def find_repo_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd()]:
        if (candidate / 'data' / 'features' / 'player_defensive_actions.parquet').exists():
            return candidate
    raise FileNotFoundError('Could not locate data/features/player_defensive_actions.parquet from the current working directory.')

def mode_or_unknown(s: pd.Series) -> str:
    s = s.dropna()
    return s.mode().iloc[0] if not s.empty else 'Unknown'

REPO_ROOT = find_repo_root()
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
DATA_RAW = REPO_ROOT / 'data' / 'raw'

df = pd.read_parquet(DATA_FEATURES / 'player_defensive_actions.parquet')

# Match metadata is optional for the clustering, but it makes the profile unit clearer.
match_paths = sorted((DATA_RAW / 'matches').glob('*.json'))
if match_paths:
    matches = pd.concat([pd.read_json(path) for path in match_paths], ignore_index=True)
    meta_cols = [c for c in ['match_id', 'competition_name', 'season', 'competition_stage'] if c in matches.columns]
    meta = matches[meta_cols].drop_duplicates('match_id').copy()
    if {'competition_name', 'season'}.issubset(meta.columns):
        meta['tournament'] = meta['competition_name'].astype(str) + ' ' + meta['season'].astype(str)
    else:
        meta['tournament'] = 'Unknown tournament'
    if 'competition_stage' in meta.columns:
        meta['stage_group'] = np.where(meta['competition_stage'].eq('Group Stage'), 'Group Stage', 'Knockout')
    df = df.merge(meta[['match_id', 'tournament', 'stage_group']], on='match_id', how='left')
else:
    df['tournament'] = 'All matches'
    df['stage_group'] = 'Unknown'

df['tournament'] = df['tournament'].fillna('Unknown tournament')
df['team_profile'] = df['tournament'].astype(str) + ' — ' + df['team'].astype(str)

print(f'Repository root: {REPO_ROOT}')
print(f'Defensive actions: {len(df):,}')
print(f'Teams: {df["team"].nunique()}')
print(f'Team-tournament profiles: {df["team_profile"].nunique()}')
display(df[['tournament', 'team', 'phase_label', 'action_family', 'target_shot_in_10s', 'target_xt_10s']].head())

---
## 1. Build team-tournament profile features

Profiles combine:
- **Volume / intensity:** defensive actions per match
- **Outcome risk:** downstream shot rate and mean xT
- **Phase identity:** high press, counterpress, low block, box defending, wide defending, central progression defending
- **Action identity:** pressure, contests, recoveries, interventions
- **Spatial footprint:** central/wide/deep/high lane shares and mean distance to nearest goal
- **360 support:** support balance and nearest opponent context

In [ ]:
profile_key = ['tournament', 'team']

base = (
    df.groupby(profile_key)
    .agg(
        matches=('match_id', 'nunique'),
        actions=('event_id', 'size'),
        players=('player_id', 'nunique'),
        shot_rate=('target_shot_in_10s', 'mean'),
        mean_xt=('target_xt_10s', 'mean'),
        has_360_share=('has_360', 'mean'),
        counterpress_share=('counterpress', 'mean'),
        central_lane_share=('is_central_lane', 'mean'),
        wide_lane_share=('is_wide_lane', 'mean'),
        deep_zone_share=('is_deep_zone', 'mean'),
        high_zone_share=('is_high_zone', 'mean'),
        avg_goal_distance=('nearest_goal_distance', 'mean'),
        avg_support_balance_10m=('freeze_support_balance_10m', 'mean'),
        avg_support_ratio_10m=('freeze_support_ratio_10m', 'mean'),
        avg_opponent_nearest_distance=('freeze_opponent_nearest_distance', 'mean'),
    )
    .reset_index()
)
base['actions_per_match'] = base['actions'] / base['matches'].clip(lower=1)

phase_share = pd.crosstab([df['tournament'], df['team']], df['phase_label'], normalize='index').add_prefix('phase__')
action_share = pd.crosstab([df['tournament'], df['team']], df['action_family'], normalize='index').add_prefix('action__')
zone_share = pd.crosstab([df['tournament'], df['team']], df['action_zone'], normalize='index').add_prefix('zone__')

team_profiles = (
    base.set_index(profile_key)
    .join(phase_share, how='left')
    .join(action_share, how='left')
    .join(zone_share, how='left')
    .fillna(0)
    .reset_index()
)
team_profiles['profile_name'] = team_profiles['tournament'].astype(str) + ' — ' + team_profiles['team'].astype(str)

MIN_MATCHES = 3
MIN_ACTIONS = 250
profiles = team_profiles[(team_profiles['matches'] >= MIN_MATCHES) & (team_profiles['actions'] >= MIN_ACTIONS)].copy()

print(f'All profiles: {len(team_profiles)}')
print(f'Profiles retained (matches >= {MIN_MATCHES}, actions >= {MIN_ACTIONS}): {len(profiles)}')
display(profiles[['tournament', 'team', 'matches', 'actions', 'actions_per_match', 'shot_rate', 'mean_xt']].sort_values('shot_rate', ascending=False).round(4).head(12))

---
## 2. Choose cluster count and fit K-Means

We keep risk metrics in the feature set so the clusters represent both style and consequence. The silhouette chart is a sanity check rather than a hard rule; football interpretability still matters.

In [ ]:
candidate_features = [
    'actions_per_match', 'shot_rate', 'mean_xt', 'has_360_share', 'counterpress_share',
    'central_lane_share', 'wide_lane_share', 'deep_zone_share', 'high_zone_share',
    'avg_goal_distance', 'avg_support_balance_10m', 'avg_support_ratio_10m',
    'avg_opponent_nearest_distance',
    'phase__high_press', 'phase__counterpress_after_loss', 'phase__settled_low_block',
    'phase__settled_mid_block', 'phase__box_defence', 'phase__wide_defending',
    'phase__central_progression_defence', 'phase__second_ball',
    'action__pressure', 'action__contest', 'action__recovery', 'action__intervention',
    'action__goalkeeper', 'action__discipline',
]
feature_cols = [c for c in candidate_features if c in profiles.columns]

X = profiles[feature_cols].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

max_k = min(7, len(profiles) - 1)
k_results = []
for k in range(2, max_k + 1):
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=50).fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels) if len(set(labels)) > 1 else np.nan
    k_results.append({'k': k, 'silhouette': score})
k_results = pd.DataFrame(k_results)
display(k_results.round(4))

if len(k_results):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=k_results, x='k', y='silhouette', marker='o', ax=ax)
    ax.set_title('Silhouette score by cluster count')
    ax.set_xlabel('Number of clusters')
    ax.set_ylabel('Silhouette score')
    plt.tight_layout()
    plt.show()

# Four clusters usually gives interpretable tactical families for this data size.
N_CLUSTERS = min(4, max(2, len(profiles) // 4))
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=100)
profiles['cluster'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=feature_cols)
centers['cluster'] = range(N_CLUSTERS)
display(centers.set_index('cluster').round(4))

---
## 3. Give clusters analyst-friendly labels

Labels are assigned from cluster centroids using clear heuristics. The table keeps the numeric cluster IDs for reproducibility.

In [ ]:
center_matrix = centers.set_index('cluster').copy()
center_z = (center_matrix - center_matrix.mean()) / center_matrix.std(ddof=0).replace(0, 1)

style_components = {
    'Front-foot pressers': ['phase__high_press', 'phase__counterpress_after_loss', 'counterpress_share'],
    'Deep-block absorbers': ['phase__settled_low_block', 'phase__box_defence', 'deep_zone_share'],
    'High-central disruptors': ['central_lane_share', 'high_zone_share', 'phase__central_progression_defence'],
    'Wide-channel protectors': ['wide_lane_share', 'phase__wide_defending'],
    'Risk-exposed profiles': ['shot_rate', 'mean_xt'],
}

style_scores = pd.DataFrame(index=center_z.index)
for style_name, style_cols in style_components.items():
    valid_cols = [c for c in style_cols if c in center_z.columns]
    if valid_cols:
        style_scores[style_name] = center_z[valid_cols].sum(axis=1)
    else:
        style_scores[style_name] = -999.0

cluster_labels = {}
for cluster_id, score_row in style_scores.iterrows():
    ranked = score_row.sort_values(ascending=False)
    primary = ranked.index[0]
    secondary = ranked.index[1] if len(ranked) > 1 else primary
    cluster_labels[cluster_id] = f'{primary} ({secondary}-leaning)'

profiles['cluster_label'] = profiles['cluster'].map(cluster_labels)

cluster_summary = (
    profiles.groupby(['cluster', 'cluster_label'])
    .agg(
        profiles=('profile_name', 'count'),
        avg_matches=('matches', 'mean'),
        avg_actions_per_match=('actions_per_match', 'mean'),
        avg_shot_rate=('shot_rate', 'mean'),
        avg_xt=('mean_xt', 'mean'),
        high_press=('phase__high_press', 'mean'),
        counterpress=('counterpress_share', 'mean'),
        low_block=('phase__settled_low_block', 'mean'),
        box_defence=('phase__box_defence', 'mean'),
        central_lane=('central_lane_share', 'mean'),
        wide_lane=('wide_lane_share', 'mean'),
    )
    .reset_index()
    .sort_values('avg_shot_rate')
)
display(cluster_summary.round(4))

---
## 4. Cluster map and centroid heatmap

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
profiles['pc1'] = coords[:, 0]
profiles['pc2'] = coords[:, 1]

fig, ax = plt.subplots(figsize=(13, 9))
sns.scatterplot(
    data=profiles,
    x='pc1', y='pc2',
    hue='cluster_label',
    size='shot_rate',
    sizes=(80, 260),
    alpha=0.85,
    ax=ax,
)
for _, row in profiles.iterrows():
    label = f"{row['team']} ({str(row['tournament']).split()[-1]})"
    ax.text(row['pc1'] + 0.04, row['pc2'] + 0.04, label, fontsize=8)
ax.set_title(f'Team defensive profile clusters (PCA explains {pca.explained_variance_ratio_.sum()*100:.1f}% of scaled variance)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

heat_cols = [c for c in [
    'actions_per_match', 'shot_rate', 'mean_xt', 'counterpress_share', 'central_lane_share', 'deep_zone_share',
    'high_zone_share', 'phase__high_press', 'phase__settled_low_block', 'phase__box_defence',
    'phase__wide_defending', 'action__pressure', 'action__contest', 'action__recovery',
] if c in centers.columns]
center_plot = centers.set_index('cluster')[heat_cols]
center_plot.index = [f"{idx}: {cluster_labels[idx]}" for idx in center_plot.index]
center_z = (center_plot - center_plot.mean()) / center_plot.std(ddof=0).replace(0, 1)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(center_z, cmap='vlag', center=0, annot=center_plot.round(3), fmt='', ax=ax)
ax.set_title('Cluster centroids: raw values annotated, color shows relative z-score')
plt.tight_layout()
plt.show()

---
## 5. Team tables: safest, riskiest, and strongest representatives

In [ ]:
display_cols = ['tournament', 'team', 'cluster_label', 'matches', 'actions_per_match', 'shot_rate', 'mean_xt', 'phase__high_press', 'phase__settled_low_block', 'phase__box_defence', 'counterpress_share']
display_cols = [c for c in display_cols if c in profiles.columns]

print('Lowest downstream shot rates among retained team profiles')
display(profiles.sort_values('shot_rate')[display_cols].head(12).round(4))

print('Highest downstream shot rates among retained team profiles')
display(profiles.sort_values('shot_rate', ascending=False)[display_cols].head(12).round(4))

print('Most representative profiles per cluster (closest to centroid in scaled feature space)')
distances = kmeans.transform(X_scaled)
profiles['distance_to_cluster_center'] = [distances[i, c] for i, c in enumerate(profiles['cluster'])]
representatives = (
    profiles.sort_values('distance_to_cluster_center')
    .groupby('cluster_label', as_index=False)
    .head(5)
    .sort_values(['cluster_label', 'distance_to_cluster_center'])
)
display(representatives[['cluster_label', 'tournament', 'team', 'matches', 'shot_rate', 'mean_xt', 'distance_to_cluster_center']].round(4))

---
## 6. Analyst summary

In [ ]:
print('=' * 90)
print('TEAM DEFENSIVE CLUSTERING SUMMARY')
print('=' * 90)
print(f'Profiles clustered: {len(profiles)}')
print(f'Clusters: {N_CLUSTERS}')
print(f'Features used: {len(feature_cols)}')
print()

for _, row in cluster_summary.iterrows():
    subset = profiles[profiles['cluster'] == row['cluster']].sort_values('shot_rate')
    examples = ', '.join(subset['team'].head(4).tolist())
    print(f"Cluster {int(row['cluster'])}: {row['cluster_label']}")
    print(f"  Profiles: {int(row['profiles'])}; avg shot rate: {row['avg_shot_rate']*100:.2f}%; avg xT: {row['avg_xt']:.3f}")
    print(f"  Style: high press {row['high_press']*100:.1f}%, low block {row['low_block']*100:.1f}%, box defence {row['box_defence']*100:.1f}%")
    print(f"  Example lower-risk teams: {examples}")
    print()

safest = profiles.sort_values('shot_rate').iloc[0]
riskiest = profiles.sort_values('shot_rate', ascending=False).iloc[0]
print(f"Safest retained profile by shot rate: {safest['team']} ({safest['tournament']}) — {safest['shot_rate']*100:.2f}%")
print(f"Riskiest retained profile by shot rate: {riskiest['team']} ({riskiest['tournament']}) — {riskiest['shot_rate']*100:.2f}%")